In [ ]:
from pathlib import Path
import base64, hashlib, subprocess, sys
runner = Path("/kaggle/working/run_h7_kaggle.py")
reservation = Path("/kaggle/working/evaluation.reservation.json")
with runner.open("xb") as handle:
    handle.write(base64.b64decode('...'))
with reservation.open("xb") as handle:
    handle.write(base64.b64decode('ewogICJhdHRlbXB0X2lkIjogImFiZDMzYTUxLTQxOWQtNDRjOS04N2ZmLTRiMTY1YWNmZDIzNSIsCiAgImF1dGhvcml6YXRpb25fY29zdF91c2QiOiAwLAogICJjYW1wYWlnbl9ydW50aW1lX2NhcF9zZWNvbmRzIjogMTQ0MDAsCiAgImNhbm9uaWNhbF9hdXRob3JpdHkiOiAiL1VzZXJzL21hYy8uY29kZXgvbmFuby1ldmFsdWF0aW9uLWF1dGhvcml0eS9uYW5vLXAxLWg3LWJvdW5kYXJ5LW1hcmdpbi1zY3JlZW4tdjEiLAogICJleHBlcmltZW50IjogIm5hbm8tcDEtaDctYm91bmRhcnktbWFyZ2luLXNjcmVlbi12MSIsCiAgImtlcm5lbF9tZXRhZGF0YSI6IHsKICAgICJjb2RlX2ZpbGUiOiAiaDdfZXZhbHVhdGlvbi5pcHluYiIsCiAgICAiY29tcGV0aXRpb25fc291cmNlcyI6IFtdLAogICAgImRhdGFzZXRfc291cmNlcyI6IFsKICAgICAgImhhc3NhbmVsamVzci9uYW5vLWg3LWV2YWx1YXRpb24tZDZlZDIxMiIsCiAgICAgICJoYXNzYW5lbGplc3IvbmFuby1oNy10cmVhdG1lbnRzLWM2MTc3MDY3IgogICAgXSwKICAgICJkb2NrZXJfaW1hZ2UiOiAiZ2NyLmlvL2thZ2dsZS1pbWFnZXMvcHl0aG9uQHNoYTI1Njo4MWIxZTdjNGI0ZjBmMmE3ZTBhMzM5NDNjOTU3NTRiMWI3YWY2YjMwZDY1MjIxNjMzYTI4NGI2YjYwMWNkODRhIiwKICAgICJkb2NrZXJfaW1hZ2VfcGlubmluZ190eXBlIjogIm9yaWdpbmFsIiwKICAgICJlbmFibGVfZ3B1IjogZmFsc2UsCiAgICAiZW5hYmxlX2ludGVybmV0IjogdHJ1ZSwKICAgICJlbmFibGVfdHB1IjogZmFsc2UsCiAgICAiaWQiOiAiaGFzc2FuZWxqZXNyL25hbm8taDctZXZhbC1ydW4tZDZlZDIxMiIsCiAgICAiaXNfcHJpdmF0ZSI6IHRydWUsCiAgICAia2VybmVsX3NvdXJjZXMiOiBbXSwKICAgICJrZXJuZWxfdHlwZSI6ICJub3RlYm9vayIsCiAgICAia2V5d29yZHMiOiBbXSwKICAgICJsYW5ndWFnZSI6ICJweXRob24iLAogICAgIm1hY2hpbmVfc2hhcGUiOiAiTm9uZSIsCiAgICAibW9kZWxfc291cmNlcyI6IFtdLAogICAgInRpdGxlIjogIm5hbm8taDctZXZhbC1ydW4tZDZlZDIxMiIKICB9LAogICJtYXhfcnVudGltZV9zZWNvbmRzIjogMTA3MDUsCiAgInBhY2thZ2UiOiB7CiAgICAiYXJjaGl2ZV9zaGEyNTYiOiAiYzJjN2ZiZmU0OWFlYTE4M2RiMTFjMmMyYzE1ZWU5NzA1YjU3ZDM4ZWQ3MmQ1Y2RlNWE4NDdjNzZjYzlkMTZlYyIsCiAgICAibWFuaWZlc3Rfc2hhMjU2IjogImZjZWUyM2RjZmE0ZTE3M2JkMTEwZTE3MTdjOWM1Y2JhNTU0MzM1ZjJiNWVhZmJiMzA0OWZmNTAxOGYxYTNiN2MiLAogICAgInJlc3VsdCI6ICJuYW5vLWg3LXNjcmVlbi1yZXN1bHRzLnRhci5neiIsCiAgICAicm9vdCI6ICJuYW5vLWg3LWV2YWx1YXRpb24iLAogICAgInNjcmlwdCI6ICJSVU5fSDdfRVZBTFVBVEUuc2giCiAgfSwKICAicmVzZXJ2ZWRfYXRfdXRjIjogIjIwMjYtMTAtMjJUMjA6Mzk6MjVaIiwKICAicmV0cnlfcG9saWN5IjogIk5ldmVyIGF1dG9tYXRpY2FsbHkgcmV0cnkgb3IgcmVsZWFzZTsgdGVybWluYWwgb3IgdW5rbm93biBhdHRlbXB0cyByZXF1aXJlIHJldmlldy4iLAogICJydW5uZXJfc2hhMjU2IjogImYwZDdlZWQ5OWY2YjFlYWYyOGUyMTRjMGU3MzkyNDU4NWVjOTFiOTIzMzMzNzlhNDYwZjJkNGQ1M2Y4ZDhjODYiLAogICJzY2hlbWEiOiAibmFuby5oNy1vcGVyYXRpb25hbC1yZXNlcnZhdGlvbi52MSIsCiAgInN0YWdlIjogImV2YWx1YXRpb24iLAogICJzdGFnZV9kYXRhc2V0IjogImhhc3NhbmVsamVzci9uYW5vLWg3LWV2YWx1YXRpb24tZDZlZDIxMiIsCiAgInRyYWluaW5nX3Rlcm1pbmFsIjogewogICAgImF0dGVtcHRfaWQiOiAiYzYxNzcwNjctNDg3Zi00ODcyLWE4NGUtY2QyYzllYWI4Mjc4IiwKICAgICJlbGFwc2VkX3NlY29uZHMiOiAzNjk0LjcwNywKICAgICJlcnJvciI6IG51bGwsCiAgICAiZXhpdF9jb2RlIjogMCwKICAgICJmaW5pc2hlZF9hdF91dGMiOiAiMjAyNi0xMC0yMlQyMDozNToxMVoiLAogICAgInBoYXNlIjogImNvbXBsZXRlZCIsCiAgICAicmVzZXJ2YXRpb25fc2hhMjU2IjogImI1N2Q4NjgwZDA4MzJjM2UwZjZiZTdjZjJlOTIxZDBiMzdjODU4NzkxZjI4YWVmZjM5ZTFhM2I1NzMxMTBiNDciLAogICAgInJldHJ5X2F1dGhvcml6ZWQiOiBmYWxzZSwKICAgICJzdGFnZSI6ICJ0cmFpbmluZyIsCiAgICAic3RhcnRlZF9hdF91dGMiOiAiMjAyNi0xMC0yMlQxOTozMzozNloiLAogICAgInN0YXR1cyI6ICJjb21wbGV0ZWQiCiAgfSwKICAidHJlYXRtZW50X2FyY2hpdmVfc2hhMjU2IjogImNlZjUzZmZlNjgyMmNkZmQ1YmIzMWQwMzU3NWI5OGE2OTBlZmU0ZDFhYjIyMzA4OGJiMTVlZGUwM2U5MDM0MTQiLAogICJ0cmVhdG1lbnRfZGF0YXNldCI6ICJoYXNzYW5lbGplc3IvbmFuby1oNy10cmVhdG1lbnRzLWM2MTc3MDY3IiwKICAidmVyaWZpZWRfcHJpdmF0ZV9kYXRhc2V0cyI6IFsKICAgICJoYXNzYW5lbGplc3IvbmFuby1oNy1ldmFsdWF0aW9uLWQ2ZWQyMTIiLAogICAgImhhc3NhbmVsamVzci9uYW5vLWg3LXRyZWF0bWVudHMtYzYxNzcwNjciCiAgXQp9Cg=='))
manifests = list(Path("/kaggle/input").rglob("BUNDLE_MANIFEST.json"))
assert len(manifests) == 1, manifests
assert hashlib.sha256(manifests[0].read_bytes()).hexdigest() == "fcee23dcfa4e173bd110e1717c9c5cba554335f2b5eafbb3049ff5018f1a3b7c"
treatments = list(Path("/kaggle/input").rglob("nano-h7-treatment-results.tar.gz.bin"))
assert len(treatments) == 1, treatments
subprocess.run([sys.executable, "-u", str(runner), "run", "--reservation", str(reservation), "--source-dir", str(manifests[0].parent), "--output-dir", "/kaggle/working/h7-evaluation", "--treatment-archive", str(treatments[0])], check=True)
